In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore') 


datadir = 'dados'

# 1.1 CARGA DAS TABELAS
print("Carregando os dados...")
discentes = pd.read_parquet(os.path.join(datadir, 'discentes.parquet'))
# Garantir unicidade na tabela de discentes
discentes.drop_duplicates(subset=['id_discente'], inplace=True, ignore_index=True)

matriculas = pd.read_parquet(os.path.join(datadir, 'matriculas.parquet'))
componentes = pd.read_parquet(os.path.join(datadir, 'componentes.parquet'))
cursos = pd.read_parquet(os.path.join(datadir, 'cursos.parquet'))

print(f"Total original de Discentes: {len(discentes)}")
print(f"Total de Matrículas: {len(matriculas)}")

# 1.2 DEFINIÇÃO DO TARGET (VARIÁVEL ALVO)
# Remover alunos sem status definido
discentes = discentes.dropna(subset=['status_discente'])

# Criar a variável 'evasao' (1 = EVADIDO/CANCELADO, 0 = ATIVO/CONCLUÍDO/FORMADO)
status_evasao = ['EVADIDO', 'CANCELADO']
discentes['evasao'] = discentes['status_discente'].apply(lambda x: 1 if x in status_evasao else 0)

# 1.3 MERGE INICIAL 
# Juntando o histórico de matrículas com as informações do aluno, disciplinas e cursos
df_unificado = (
    matriculas
    .merge(discentes, how='inner', on='id_discente')
    .merge(componentes, how='left', on='id_disciplina')
    .merge(cursos, how='left', on=['id_curso', 'id_estrutura_curricular', 'id_disciplina'])
)

print(f"Base unificada criada com {len(df_unificado)} linhas e {df_unificado.shape[1]} colunas.")
print(f"Proporção de Evasão Geral na base de discentes: {discentes['evasao'].mean():.1%}")

In [ ]:
print("Iniciando a Engenharia de Variáveis (Feature Engineering)...")

# 2.1 Evitando Vazamento de Dados (Foco no 1º Período Letivo)
print("Filtrando apenas o 1º período letivo para evitar Data Leakage...")

# Identificar o nome correto da coluna 'periodo' (pode ter sido renomeada no merge da Parte 1)
col_periodo = 'periodo_x' if 'periodo_x' in df_unificado.columns else 'periodo'

# Identificar o primeiro ano e período de cada aluno
primeiro_periodo_df = df_unificado.groupby('id_discente')[['ano', col_periodo]].min().reset_index()

# Filtrar a base unificada apenas para as interações desse primeiro período
status_1o_semestre = df_unificado.merge(primeiro_periodo_df, on=['id_discente', 'ano', col_periodo])

# 2.2 Criando Histórico de Aprovações e Reprovações do 1º Semestre
hist_disc = (
    status_1o_semestre[['situacao', 'id_disciplina', 'id_discente']]
    .assign(
        aprov_1o_sem = lambda x: x['situacao'].str.contains("^APROV", na=False).astype(int),
        reprov_1o_sem = lambda x: x['situacao'].str.contains("^REP", na=False).astype(int)
    )
    .groupby(['id_discente'])
    .agg({"aprov_1o_sem": "sum", "reprov_1o_sem": "sum"})
    .reset_index()
)

# 2.3 Criando Features Demográficas e de Ingresso
df = df_unificado.copy()
df["idade_ingresso"] = df["ano_ingresso"] - df["ano_nascimento"]
df["ingresso_recente"] = (df["ano_ingresso"] >= 2020).astype(int)

# 2.4 Consolidando a Tabela de Features (Matriz X)
x1 = (
    df.dropna(subset=['idade_ingresso']) # Remover alunos sem idade definida
    .drop_duplicates(subset='id_discente', ignore_index=True) # Manter apenas uma linha por aluno
    .merge(hist_disc, on='id_discente', how='left') # Juntar o histórico calculado do 1º semestre
)

# Alunos sem notas no 1º semestre recebem 0 aprovações/reprovações
x1['aprov_1o_sem'] = x1['aprov_1o_sem'].fillna(0)
x1['reprov_1o_sem'] = x1['reprov_1o_sem'].fillna(0)

# 2.5 Tratamento de Variáveis Categóricas (Criação de Dummies)
def gera_dummies(df, colname, map_dict=None):
    if map_dict is not None:
        df[colname] = df[colname].map(map_dict)
    else:
        df[colname] = df[colname].astype(str).str.lower()
    return pd.get_dummies(df, columns=[colname], drop_first=True, dtype=int)

# Transformando texto em colunas binárias (0 ou 1)
x1 = gera_dummies(x1, 'sexo',  {"F": "feminino", "M": "masculino"})
x1 = gera_dummies(x1, 'estado_civil',  {"Solteiro(a)": "solteiro", "Casado(a)": "casado"})

for col in ['raca_declarada', 'faixa_renda_familiar', 'id_curso']:
    x1 = gera_dummies(x1, col)

# 2.6 Seleção Final das Colunas que irão para o Modelo
colunas_base = ['evasao', 'ano_ingresso', 'periodo_ingresso', 'idade_ingresso',
                'aprov_1o_sem', 'reprov_1o_sem', 'ingresso_recente']

# Pega automaticamente todas as dummies geradas
colunas_dummies = [c for c in x1.columns if c.startswith(("sexo_", "estado_civil_", "raca_", "faixa_", "id_curso_"))]

data = x1[colunas_base + colunas_dummies]

print(f"Feature Engineering concluída! Base pronta para modelagem tem {len(data)} linhas e {data.shape[1]} colunas.")

In [ ]:
print("Iniciando a Parte 3: Separação de Treino/Teste e Treinamento dos Modelos...")

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import numpy as np

# 3.1 Separação de Treino e Teste (Corte Temporal)
CUTOFF = 2020
ANO_ATUAL = data['ano_ingresso'].max()

# Remover alunos dos últimos 2 anos (para não enviesar a base, pois eles ainda não tiveram tempo de evadir ou se formar)
data_model = data[data["ano_ingresso"] <= ANO_ATUAL - 2].copy()

# Treino com dados históricos (até 2020) e Teste com dados "futuros" (após 2020)
train = data_model[data_model["ano_ingresso"] <= CUTOFF].copy()
test  = data_model[data_model["ano_ingresso"] > CUTOFF].copy()

X_train = train.drop(['evasao'], axis=1)
y_train = train['evasao']
X_test = test.drop(['evasao'], axis=1)
y_test = test['evasao']

print(f"Treino: {len(train)} linhas ({len(train)/len(data_model):.1%}) | Proporção de Evasão: {y_train.mean():.1%}")
print(f"Teste: {len(test)} linhas ({len(test)/len(data_model):.1%}) | Proporção de Evasão: {y_test.mean():.1%}")

# 3.2 Configuração da Validação Cruzada (Cross-Validation com 5 Folds)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# =========================================================
# 3.3 CONSTRUÇÃO DOS PIPELINES E TREINAMENTO (TUNING)
# =========================================================

# A) Regressão Logística (Com balanceamento de classes e regularização L1/L2)
pipe_logit = Pipeline([
    ("imputer", SimpleImputer(strategy="median")), # Preenche NAs
    ("scaler", StandardScaler()), # Padroniza os dados 
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", solver="saga", random_state=42))
])
param_logit = {"model__C": [0.1, 1], "model__penalty": ["l1", "l2"]}

print("\nTreinando Regressão Logística (Pode levar alguns segundos)...")
grid_logit = GridSearchCV(pipe_logit, param_grid=param_logit, cv=cv, scoring='roc_auc', n_jobs=-1)
grid_logit.fit(X_train, y_train)
print(f"Logit ROC-AUC (Treino/Validação): {grid_logit.best_score_:.4f}")


# B) Random Forest (Floresta Aleatória)
pipe_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(random_state=42, class_weight="balanced"))
])

param_rf = {"model__n_estimators": [100, 200], "model__max_depth": [3, 5, 7]}

print("\nTreinando Random Forest...")
grid_rf = GridSearchCV(pipe_rf, param_grid=param_rf, cv=cv, scoring='roc_auc', n_jobs=-1)
grid_rf.fit(X_train, y_train)
print(f"Random Forest ROC-AUC (Treino/Validação): {grid_rf.best_score_:.4f}")


# C) XGBoost 
# Calculando o peso das classes para balanceamento no XGBoost
peso_classes = (len(y_train) - y_train.sum()) / y_train.sum()

pipe_xgb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(random_state=42, eval_metric="logloss", scale_pos_weight=peso_classes))
])

param_xgb = {"model__n_estimators": [100, 200], "model__max_depth": [3, 5], "model__learning_rate": [0.05, 0.1]}

print("\nTreinando XGBoost...")
grid_xgb = GridSearchCV(pipe_xgb, param_grid=param_xgb, cv=cv, scoring='roc_auc', n_jobs=-1)
grid_xgb.fit(X_train, y_train)
print(f"XGBoost ROC-AUC (Treino/Validação): {grid_xgb.best_score_:.4f}")

# =========================================================
# 3.4 SELEÇÃO DO MELHOR MODELO
# =========================================================
historico_modelos = {
    "Regressão Logística": grid_logit,
    "Random Forest": grid_rf,
    "XGBoost": grid_xgb
}

# Escolhe automaticamente o modelo com a maior área sob a curva (ROC-AUC)
nome_melhor_modelo = max(historico_modelos, key=lambda k: historico_modelos[k].best_score_)
melhor_modelo = historico_modelos[nome_melhor_modelo].best_estimator_

print(f"\n✅ Melhor modelo selecionado: {nome_melhor_modelo}")

In [ ]:
print("A iniciar a Parte 4: Avaliação Focada na Evasão (Ajuste de Recall)...")

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 4.1 Prever probabilidades na base de teste (alunos pós-2020)
# Usamos [:, 1] para extrair apenas a probabilidade de o aluno pertencer à classe 1 (Evasão)
y_probs = melhor_modelo.predict_proba(X_test)[:, 1]

# 4.2 Avaliação padrão (Threshold 0.5)
print("=== DESEMPENHO PADRÃO (Limiar de 0.5) ===")
y_pred_default = (y_probs >= 0.5).astype(int)
print(classification_report(y_test, y_pred_default))

# =========================================================
# 4.3 AJUSTE DE THRESHOLD PARA MAXIMIZAR O RECALL
# =========================================================
# Para um sistema de alerta antecipado, precisamos priorizar não perder os alunos em risco.
# Vamos procurar o limiar exato que garanta um Recall alto (neste caso, tentaremos garantir pelo menos 85 a 88%)

precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)

# Definimos o alvo de captura de evasões em 88%
target_recall = 0.88

# Encontrar o índice do primeiro recall que atenda ou supere o nosso alvo
idx = np.where(recalls >= target_recall)[0][-1]
best_threshold = thresholds[idx]

print(f"\n=== DESEMPENHO OTIMIZADO PARA ALERTA DA GESTÃO ===")
print(f"Limiar de decisão ajustado de 0.5000 para: {best_threshold:.4f}")

# Gerar as novas previsões com o limiar otimizado
y_pred_otimizado = (y_probs >= best_threshold).astype(int)
print(classification_report(y_test, y_pred_otimizado))

# Métrica global de separabilidade
roc_auc = roc_auc_score(y_test, y_probs)
print(f"ROC-AUC no Teste: {roc_auc:.4f}")

# =========================================================
# 4.4 VISUALIZAÇÃO DA MATRIZ DE CONFUSÃO
# =========================================================
plt.figure(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred_otimizado)

# Criar o mapa de calor (heatmap) para visualizar os acertos e erros
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Ativo/Formado (0)', 'Evasão (1)'],
            yticklabels=['Ativo/Formado (0)', 'Evasão (1)'])

plt.ylabel('Realidade (O que aconteceu de facto)')
plt.xlabel('Previsão do Modelo (O que o sistema alertou)')
plt.title(f'Matriz de Confusão - Modelo Otimizado\\n(Threshold: {best_threshold:.4f} | ROC-AUC: {roc_auc:.4f})')
plt.tight_layout()
plt.show()

In [ ]:
print("Iniciando a Explicabilidade do Modelo com SHAP...")

import shap
import matplotlib.pyplot as plt


imputer = melhor_modelo.named_steps['imputer']
estimador_final = melhor_modelo.named_steps['model']

# Aplicar a transformação (preenchimento de NAs) nos dados de teste
X_test_imputed = imputer.transform(X_test)
# Converter de volta para DataFrame para manter os nomes das colunas nos gráficos
X_test_imputed_df = pd.DataFrame(X_test_imputed, columns=X_test.columns)

# Inicializar o explainer do SHAP para modelos baseados em árvores (como o XGBoost e Random Forest)
explainer = shap.TreeExplainer(estimador_final)
shap_values = explainer.shap_values(X_test_imputed_df)

# =========================================================
# GRÁFICO 1: Importância Geral das Variáveis (Feature Importance)
# =========================================================
plt.figure(figsize=(10, 6))
plt.title("Quais fatores mais influenciam a previsão do modelo? (SHAP Bar Plot)")
shap.summary_plot(shap_values, X_test_imputed_df, plot_type="bar", show=False)
plt.tight_layout()
plt.show()

# =========================================================
# GRÁFICO 2: Impacto Direto na Evasão (Positivo ou Negativo)
# =========================================================
# Este gráfico mostra se valores altos/baixos da variável empurram a previsão para Evasão (1) ou Retenção (0)
plt.figure(figsize=(10, 6))
plt.title("Impacto Direto das Variáveis na Evasão (SHAP Summary)")
shap.summary_plot(shap_values, X_test_imputed_df, show=False)
plt.tight_layout()
plt.show()

In [ ]:
print("Iniciando a Parte 5: Salvando o Modelo para Deploy...")
import pickle


output = {
    "modelo": melhor_modelo,
    "threshold": float(best_threshold), 
    "features": list(X_train.columns),
    "nome_modelo": nome_melhor_modelo
}


with open("modelo_evasao_ufpb.pkl", "wb") as file:
    pickle.dump(output, file)

print("✅ Modelo e limiar otimizado salvos com sucesso no arquivo 'modelo_evasao_ufpb.pkl'!")